In [ ]:
#%pip install torch -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
#%pip install torch-geometric -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
#%pip install torch-scatter -f https://data.pyg.org/whl/torch-2.7.0+cu126.html
#%pip install torch-sparse -f https://data.pyg.org/whl/torch-2.7.0+cu126.html
#%pip install test_tube
#%pip install optuna

# Optuna Search for KarateLink

In [ ]:
from model import add_model_args
from link_prediction import main
import optuna
from optuna.trial import Trial
import pickle

def load_shared_resources(dataset_name):
    dataset_path = '{dataset_name}.pkl'.format(dataset_name=dataset_name)
    splits_path = '{dataset_name}_splits.pkl'.format(dataset_name=dataset_name)
    with open(dataset_path, 'rb') as f:
        dataset = pickle.load(f)
    with open(splits_path, 'rb') as f:
        splits = pickle.load(f)
    return dataset, splits

def objective(trial: Trial):
    parser = add_model_args(None, hyper=False)

    # Suggest hyperparameters
    args = parser.parse_args([])  # Empty because we'll override manually
    args.threshold = trial.suggest_float('threshold', 0.1, 0.9)
    args.num_agents = trial.suggest_categorical('num_agents', [4, 6])
    args.batch_size = trial.suggest_categorical('batch_size', [16, 32])
    args.hidden_units = trial.suggest_categorical('hidden_units', [8, 16, 32])
    args.num_steps = trial.suggest_categorical('num_steps', [3, 6, 9])
    args.lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    args.dropout = trial.suggest_float('dropout', 0.0, 0.5)

    args.num_pos_attention_heads = trial.suggest_categorical('num_pos_attention_heads', [1, 2, 4])
    args.gumbel_temp = trial.suggest_float('gumbel_temp', 0.3, 1.0)
    
    args.readout_type = trial.suggest_categorical('readout_type', ['final_only', 'all_steps', 'all_steps_linear', 'all_steps_mlp'])

    # Fixed settings
    args.epochs = 200
    args.dataset = 'KarateLink'
    args.reduce = 'sum'
    args.global_agent_pool = True
    args.agent_global_extra = True
    args.bias_attention = True
    args.basic_agent = True
    # args.n_splits = 2 # we have already split and saved the dataset
    
    dataset, splits = load_shared_resources(args.dataset)

    # Run the model and return best validation f1 score
    return main(args, dataset, splits)['f1']


storage = "sqlite:///optuna_shared_KarateLink.db"  
study = optuna.create_study(direction='maximize', study_name='AgentNet-Tuning-KarateLink', storage=storage, load_if_exists=True)
study.optimize(objective, n_trials=30)

[I 2025-06-05 00:49:43,620] Using an existing study with name 'AgentNet-Tuning-KarateLink' instead of creating a new one.


Namespace(verbose=False, dataset='KarateLink', iters_per_epoch=30, n_splits=10, threshold=0.28719179566268593, k_hop=3, dropout=0.18668904501513928, batch_size=16, hidden_units=16, lr=0.0015277478604169485, num_agents=4, num_steps=9, reduce='sum', epochs=200, warmup=5, self_loops=False, node_readout=False, use_step_readout_lin=False, gumbel_temp=0.6921245895871533, gumbel_min_temp=0.0625, gumbel_warmup=-1, gumbel_decay_epochs=100, min_lr_mult=1e-07, weight_decay=1e-08, num_pos_attention_heads=1, readout_mlp=False, post_ln=False, attn_dropout=0.0, no_time_cond=False, mlp_width_mult=1, activation_function='leaky_relu', negative_slope=0.01, input_mlp=False, attn_width_mult=1, random_agent=False, test_argmax=False, global_agent_pool=True, agent_global_extra=True, basic_global_agent=False, basic_agent=True, bias_attention=True, visited_decay=0.9, sparse_conv=False, mean_pool_only=False, edge_negative_slope=0.2, final_readout_only=True, readout_type='final_only')
Device: cuda
LinkPredictionA

[I 2025-06-05 01:11:08,711] Trial 16 finished with value: 0.805108359133127 and parameters: {'threshold': 0.28719179566268593, 'num_agents': 4, 'batch_size': 16, 'hidden_units': 16, 'num_steps': 9, 'lr': 0.0015277478604169485, 'dropout': 0.18668904501513928, 'num_pos_attention_heads': 1, 'gumbel_temp': 0.6921245895871533, 'readout_type': 'final_only'}. Best is trial 11 with value: 0.8134817752020742.


Namespace(verbose=False, dataset='KarateLink', iters_per_epoch=30, n_splits=10, threshold=0.25104415167374494, k_hop=3, dropout=0.019073569310013472, batch_size=16, hidden_units=16, lr=0.0022198546616187943, num_agents=4, num_steps=9, reduce='sum', epochs=200, warmup=5, self_loops=False, node_readout=False, use_step_readout_lin=False, gumbel_temp=0.6911621744002427, gumbel_min_temp=0.0625, gumbel_warmup=-1, gumbel_decay_epochs=100, min_lr_mult=1e-07, weight_decay=1e-08, num_pos_attention_heads=2, readout_mlp=False, post_ln=False, attn_dropout=0.0, no_time_cond=False, mlp_width_mult=1, activation_function='leaky_relu', negative_slope=0.01, input_mlp=False, attn_width_mult=1, random_agent=False, test_argmax=False, global_agent_pool=True, agent_global_extra=True, basic_global_agent=False, basic_agent=True, bias_attention=True, visited_decay=0.9, sparse_conv=False, mean_pool_only=False, edge_negative_slope=0.2, final_readout_only=False, readout_type='all_steps')
Device: cuda
LinkPrediction

[I 2025-06-05 01:34:51,447] Trial 17 finished with value: 0.832716049382716 and parameters: {'threshold': 0.25104415167374494, 'num_agents': 4, 'batch_size': 16, 'hidden_units': 16, 'num_steps': 9, 'lr': 0.0022198546616187943, 'dropout': 0.019073569310013472, 'num_pos_attention_heads': 2, 'gumbel_temp': 0.6911621744002427, 'readout_type': 'all_steps'}. Best is trial 17 with value: 0.832716049382716.


...

Namespace(verbose=False, dataset='KarateLink', iters_per_epoch=30, n_splits=10, threshold=0.3316994581615448, k_hop=3, dropout=0.21795487354481902, batch_size=16, hidden_units=32, lr=0.001851954361725262, num_agents=6, num_steps=9, reduce='sum', epochs=200, warmup=5, self_loops=False, node_readout=False, use_step_readout_lin=False, gumbel_temp=0.48596718046420734, gumbel_min_temp=0.0625, gumbel_warmup=-1, gumbel_decay_epochs=100, min_lr_mult=1e-07, weight_decay=1e-08, num_pos_attention_heads=2, readout_mlp=False, post_ln=False, attn_dropout=0.0, no_time_cond=False, mlp_width_mult=1, activation_function='leaky_relu', negative_slope=0.01, input_mlp=False, attn_width_mult=1, random_agent=False, test_argmax=False, global_agent_pool=True, agent_global_extra=True, basic_global_agent=False, basic_agent=True, bias_attention=True, visited_decay=0.9, sparse_conv=False, mean_pool_only=False, edge_negative_slope=0.2, final_readout_only=False, readout_type='all_steps')
Device: cuda
LinkPredictionAg

[I 2025-06-05 06:05:39,326] Trial 29 finished with value: 0.8495077355836849 and parameters: {'threshold': 0.3316994581615448, 'num_agents': 6, 'batch_size': 16, 'hidden_units': 32, 'num_steps': 9, 'lr': 0.001851954361725262, 'dropout': 0.21795487354481902, 'num_pos_attention_heads': 2, 'gumbel_temp': 0.48596718046420734, 'readout_type': 'all_steps'}. Best is trial 25 with value: 0.853151397011046.


Namespace(verbose=False, dataset='KarateLink', iters_per_epoch=30, n_splits=10, threshold=0.3367566341941943, k_hop=3, dropout=0.2294713533865299, batch_size=16, hidden_units=32, lr=0.004031686218912062, num_agents=6, num_steps=9, reduce='sum', epochs=200, warmup=5, self_loops=False, node_readout=False, use_step_readout_lin=False, gumbel_temp=0.49753819143298306, gumbel_min_temp=0.0625, gumbel_warmup=-1, gumbel_decay_epochs=100, min_lr_mult=1e-07, weight_decay=1e-08, num_pos_attention_heads=2, readout_mlp=False, post_ln=False, attn_dropout=0.0, no_time_cond=False, mlp_width_mult=1, activation_function='leaky_relu', negative_slope=0.01, input_mlp=False, attn_width_mult=1, random_agent=False, test_argmax=False, global_agent_pool=True, agent_global_extra=True, basic_global_agent=False, basic_agent=True, bias_attention=True, visited_decay=0.9, sparse_conv=False, mean_pool_only=False, edge_negative_slope=0.2, final_readout_only=False, readout_type='all_steps')
Device: cuda
LinkPredictionAge

[I 2025-06-05 06:29:21,297] Trial 30 finished with value: 0.8315911730545877 and parameters: {'threshold': 0.3367566341941943, 'num_agents': 6, 'batch_size': 16, 'hidden_units': 32, 'num_steps': 9, 'lr': 0.004031686218912062, 'dropout': 0.2294713533865299, 'num_pos_attention_heads': 2, 'gumbel_temp': 0.49753819143298306, 'readout_type': 'all_steps'}. Best is trial 25 with value: 0.853151397011046.


Namespace(verbose=False, dataset='KarateLink', iters_per_epoch=30, n_splits=10, threshold=0.287842728595896, k_hop=3, dropout=0.15408414385487273, batch_size=16, hidden_units=32, lr=0.0018693543096069186, num_agents=6, num_steps=9, reduce='sum', epochs=200, warmup=5, self_loops=False, node_readout=False, use_step_readout_lin=False, gumbel_temp=0.3703210557623418, gumbel_min_temp=0.0625, gumbel_warmup=-1, gumbel_decay_epochs=100, min_lr_mult=1e-07, weight_decay=1e-08, num_pos_attention_heads=2, readout_mlp=False, post_ln=False, attn_dropout=0.0, no_time_cond=False, mlp_width_mult=1, activation_function='leaky_relu', negative_slope=0.01, input_mlp=False, attn_width_mult=1, random_agent=False, test_argmax=False, global_agent_pool=True, agent_global_extra=True, basic_global_agent=False, basic_agent=True, bias_attention=True, visited_decay=0.9, sparse_conv=False, mean_pool_only=False, edge_negative_slope=0.2, final_readout_only=False, readout_type='all_steps')
Device: cuda
LinkPredictionAge

# Best Parameters for GEPhilNet

In [ ]:
dataset = 'GePhilNet'

def objective(trial: Trial):
    parser = add_model_args(None, hyper=False)

    # Suggest hyperparameters
    args = parser.parse_args([])  # Empty because we'll override manually
    args.threshold = trial.suggest_float('threshold', 0.1, 0.9)
    args.num_agents = trial.suggest_categorical('num_agents', [4, 6])
    args.batch_size = trial.suggest_categorical('batch_size', [16, 32])
    args.hidden_units = trial.suggest_categorical('hidden_units', [8, 16, 32])
    args.num_steps = trial.suggest_categorical('num_steps', [3, 6, 9])
    args.lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    args.dropout = trial.suggest_float('dropout', 0.0, 0.5)

    args.num_pos_attention_heads = trial.suggest_categorical('num_pos_attention_heads', [1, 2, 4])
    args.gumbel_temp = trial.suggest_float('gumbel_temp', 0.3, 1.0)
    
    args.readout_type = trial.suggest_categorical('readout_type', ['final_only', 'all_steps', 'all_steps_linear', 'all_steps_mlp'])

    # Fixed settings
    args.epochs = 200
    args.dataset = dataset
    args.reduce = 'sum'
    args.num_edge_features = 2
    args.global_agent_pool = True
    args.agent_global_extra = True
    args.bias_attention = True
    args.basic_agent = True
    # args.n_splits = 2 # we have already split and saved the dataset
    
    dataset, splits = load_shared_resources(dataset)

    # Run the model and return best validation f1 score
    return main(args, dataset, splits)['f1']


storage = f"sqlite:///optuna_shared_{dataset}.db"  
study = optuna.create_study(direction='maximize', study_name=f'AgentNet-Tuning-{dataset}', storage=storage, load_if_exists=True)
study.optimize(objective, n_trials=60)